<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/excercise/gemini_pydantic_more_robust.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structure gemini outputs with Pydantic
- Pydantic 모델을 LLM(Gemini)의 출력 구조를 보장하는 도구로 사용하는 가장 강력하고 표준적인 방법


In [1]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('Gemini-API-Key')

from google import genai
client = genai.Client(api_key=GEMINI_API_KEY)

In [6]:
from pydantic import BaseModel

class Recipe(BaseModel):
    recipe_name: str
    description: str
    ingredients: list[str]

response = client.models.generate_content(
    model = "gemini-3.5-flash-lite",
    contents = "List a few asian soups recipes, a yummy description and list the ingredients",
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## 스키마를 이용한 LLM 출력 강제 구조화
- 프롬프트 내 지시(Text Prompting)에 의존하지 않고, Pydantic 모델을 JSON 스키마로 변환하여 Gemini API에 전달함으로써, 모델이 반드시 유효한 JSON 형식의 데이터를 생성하도록 강제

In [ ]:
Recipe(recipe_name='Sample Recipe', description='This is a sample recipe.', ingredients=['ingredient1', 'ingredient2'])

Recipe(recipe_name='Sample Recipe', description='This is a sample recipe.', ingredients=['ingredient1', 'ingredient2'])

In [5]:

response = client.models.generate_content(
    model = "gemini-2.5-flash",
    contents = "List a few asian soups recipes, a yummy description and list the ingredients",
    config = {"response_mime_type": "application/json", # 💡 출력을 JSON으로 요청
              "response_schema": list[Recipe]}          # 💡 이 JSON 스키마를 따르도록 강제
)

print(response.text)


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:
type(response.text)

str

## simulate house prices

In [ ]:
from typing import Literal

class Home(BaseModel):
    price: int
    monthly_fee: int
    living_area: float
    number_rooms: int
    type: Literal["apartment", "house"]
    address: str



response = client.models.generate_content(
    model = "gemini-2.5-flash",
    contents = """List 50 apartments and houses in Sweden with their monthly_fee, their price, living area,
    number of rooms, address, type if it is apartment or house.
    All currencies are in SEK.""",
    config = {"response_mime_type": "application/json",
              "response_schema": list[Home]}
)


homes = response.parsed
homes

[Home(price=2500000, monthly_fee=3500, living_area=75.5, number_rooms=3, type='apartment', address='Storgatan 12, Stockholm'),
 Home(price=4800000, monthly_fee=2500, living_area=120.0, number_rooms=5, type='house', address='Västra Vägen 8, Göteborg'),
 Home(price=1800000, monthly_fee=2800, living_area=50.0, number_rooms=2, type='apartment', address='Lilla Allén 5, Malmö'),
 Home(price=3200000, monthly_fee=1800, living_area=90.0, number_rooms=4, type='house', address='Gröna Gatan 15, Uppsala'),
 Home(price=2100000, monthly_fee=4200, living_area=65.0, number_rooms=3, type='apartment', address='Norra Parkvägen 7, Lund'),
 Home(price=5500000, monthly_fee=3000, living_area=150.0, number_rooms=6, type='house', address='Skogsvägen 22, Linköping'),
 Home(price=1600000, monthly_fee=3100, living_area=45.0, number_rooms=2, type='apartment', address='Östra Hamngatan 3, Helsingborg'),
 Home(price=3900000, monthly_fee=2200, living_area=105.0, number_rooms=4, type='house', address='Solrosvägen 10, Ör

In [ ]:
len(homes)

50

In [ ]:
import pandas as pd

df = pd.DataFrame([home.__dict__ for home in homes])
df.head()

,price,monthly_fee,living_area,number_rooms,type,address
0,2500000,3500,75.5,3,apartment,"Storgatan 12, Stockholm"
1,4800000,2500,120.0,5,house,"Västra Vägen 8, Göteborg"
2,1800000,2800,50.0,2,apartment,"Lilla Allén 5, Malmö"
3,3200000,1800,90.0,4,house,"Gröna Gatan 15, Uppsala"
4,2100000,4200,65.0,3,apartment,"Norra Parkvägen 7, Lund"


In [ ]:
df.query("price < 3_000_000")

,price,monthly_fee,living_area,number_rooms,type,address
0,2500000,3500,75.5,3,apartment,"Storgatan 12, Stockholm"
2,1800000,2800,50.0,2,apartment,"Lilla Allén 5, Malmö"
4,2100000,4200,65.0,3,apartment,"Norra Parkvägen 7, Lund"
6,1600000,3100,45.0,2,apartment,"Östra Hamngatan 3, Helsingborg"
8,2900000,3800,80.0,3,apartment,"Kungsgatan 1A, Jönköping"
10,2300000,3300,70.0,3,apartment,"Drottninggatan 25, Gävle"
12,1950000,2900,58.0,2,apartment,"Södergatan 9, Västerås"
14,2700000,4000,72.0,3,apartment,"Ringvägen 40, Eskilstuna"
16,1700000,3000,48.0,2,apartment,"Brunnsgatan 15, Karlstad"
18,2400000,3600,68.0,3,apartment,"Nybrogatan 8, Södertälje"
